**Data Setup and HElper Function**

In [0]:
import pandas as pd
import mlflow
import mlflow.sklearn
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score

spark.sql("USE CATALOG dev")
spark.sql("USE SCHEMA ecommerce_governed")

df_gold_spark = spark.table("gold_events")
df_gold_pandas = df_gold_spark.toPandas().fillna(0)

X = df_gold_pandas[["unique_views"]]
y = df_gold_pandas["unique_purchases"]


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)



In [0]:
models = {
    "Linear_Regression": LinearRegression(),
    "Decision_Tree": DecisionTreeRegressor(max_depth=5),
    "Random_Forest": RandomForestRegressor(n_estimators=100, max_depth=5)
}

mlflow.set_experiment("/Users/rainbow.mem5@gmail.com/Day13_Model_Tournament")

for name, model in models.items():
    with mlflow.start_run(run_name=f"Model_{name}"):
        # A. Log Hyperparameters
        mlflow.log_param("model_type", name)
        
        # B. Train
        model.fit(X_train, y_train)
        
        # C. Predict & Evaluate
        score = model.score(X_test, y_test)
        
        # D. Log Metrics & Model
        mlflow.log_metric("r2_score", score)
        mlflow.sklearn.log_model(model, "model")
        
        print(f" Model {name:20} | R2 Score: {score:.4f}")

print("\n Tournament Complete. Check MLflow UI for charts.")


In [0]:
# Spark ML Pipeline
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression as SparkLR

assembler = VectorAssembler(
    inputCols=["unique_views"], 
    outputCol="features"
)

# Stage B: Define the Model
lr_spark = SparkLR(
    featuresCol="features", 
    labelCol="unique_purchases",
    maxIter=10,
    regParam=0.3
)

pipeline = Pipeline(stages=[assembler, lr_spark])

# 3. Split Data (Native Spark Split)
train_df, test_df = df_gold_spark.fillna(0).randomSplit([0.8, 0.2], seed=42)

pipeline_model  = pipeline.fit(train_df)

# 5. Make Predictions
predictions = pipeline_model.transform(test_df)
